# 세그멘테이션 수치 검증 — 검정 · 분류 · 결정 트리

설계: [`reports/MODELING_PLAN.md`](../reports/MODELING_PLAN.md) 1·2·3단계.
입력: `data/processed/InsightStay_data_cleaned.parquet` (`scripts/build_master.py` 실행 결과 — `segment`, `L3_new`, `price_rel`, `amenity_rel`, `quote_peak` 컬럼이 있어야 합니다).

| 단계 | 질문 | 기법 |
|---|---|---|
| 1 | 층 사이 차이가 우연인가? | Kruskal-Wallis · 카이제곱 + 효과 크기 |
| 2 | 무엇이 L3(단기 무거래)를 만드나? | 로지스틱 회귀 (오즈비 · 95% 신뢰구간 · 도시 단위 검증) |
| 3 | 누구에게 개입하나? | 결정 트리 (분기점 · 안정성) |

In [ ]:
# [0-1] 라이브러리 & 기본 설정 — eda.ipynb 와 같은 스타일
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if any(_f == f.name for f in mpl.font_manager.fontManager.ttflist):
        mpl.rcParams["font.family"] = _f
        break
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.alpha"] = 0.25
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False
C = {"main": "#4E79A7", "accent": "#F28E2B", "ok": "#59A14F", "bad": "#E15759", "gray": "#9C9C9C"}

ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SEED = 0

In [ ]:
# [0-2] 데이터 로드 — 필요한 컬럼만
cols = ["id", "source", "region", "room_type", "segment", "L3_new", "minimum_nights",
        "price", "price_rel", "amenity_count", "amenity_rel", "quote_peak",
        "host_is_superhost", "number_of_reviews_ltm"]
df = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet", columns=cols)
missing = {"segment", "L3_new", "price_rel", "amenity_rel", "quote_peak"} - set(df.columns)
assert not missing, f"build_master.py 를 다시 실행하세요: {missing}"
print(f"{len(df):,}행")
df["segment"].value_counts().sort_index()

## 1. 통계 검정 — 층 차이가 우연이 아니다

- 대상: **L2~L5, `source == 0`**. L1은 99.8%가 이월값이라 특성 비교에서 뺍니다([SEGMENTATION.md](../reports/SEGMENTATION.md) L1 카드).
- 표본이 수십만 건이라 **p값은 무조건 작게 나옵니다.** 그래서 **효과 크기**를 같이 봅니다.
  - Kruskal-Wallis → ε² (0.01 작음 · 0.06 중간 · 0.14 큼)
  - 카이제곱 → Cramér's V (0.1 작음 · 0.3 중간 · 0.5 큼)

In [ ]:
# [1-1] 층별 요약
t = df[(df["source"] == 0) & df["segment"].isin(["L2", "L3", "L4", "L5"])].copy()
sh = t["host_is_superhost"].where(t["host_is_superhost"] != "unknown")   # unknown 은 분모에서 뺀다
t["superhost"] = sh.map({"t": 1, "f": 0})
t["entire"] = (t["room_type"] == "Entire home/apt").astype(int)

summary = t.groupby("segment").agg(
    n=("id", "size"),
    편의시설_중앙=("amenity_count", "median"),
    가격_중앙=("price", "median"),
    슈퍼호스트=("superhost", "mean"),
    통집=("entire", "mean"),
)
summary

In [ ]:
# [1-2] 검정 + 효과 크기
def kruskal_eps2(frame, col):
    groups = [g[col].dropna().to_numpy() for _, g in frame.groupby("segment")]
    h, p = stats.kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    return h, p, (h - k + 1) / (n - k), n

def chi2_cramer(frame, col):
    ct = pd.crosstab(frame["segment"], frame[col].dropna())
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.to_numpy().sum() * (min(ct.shape) - 1)))
    return chi2, p, v, int(ct.to_numpy().sum())

rows = []
for col, name in [("amenity_count", "편의시설 개수"), ("price", "1박 가격")]:
    h, p, e2, n = kruskal_eps2(t, col)
    rows.append({"변수": name, "검정": "Kruskal-Wallis", "통계량": h, "p": p, "효과크기": e2, "지표": "ε²", "n": n})
for col, name in [("superhost", "슈퍼호스트"), ("entire", "통집")]:
    c2, p, v, n = chi2_cramer(t, col)
    rows.append({"변수": name, "검정": "카이제곱", "통계량": c2, "p": p, "효과크기": v, "지표": "Cramér V", "n": n})
tests = pd.DataFrame(rows).set_index("변수")
tests

In [ ]:
# [1-3] 핵심 비교 L3 vs L5 — 발표에서 쓰는 쌍
l3, l5 = t[t["segment"] == "L3"], t[t["segment"] == "L5"]
pair = []
for col, name in [("amenity_count", "편의시설 개수"), ("price", "1박 가격")]:
    a, b = l3[col].dropna(), l5[col].dropna()
    u, p = stats.mannwhitneyu(a, b)
    # 순위이연상관: L3 값이 L5 값보다 클 확률 − 작을 확률
    rbc = 2 * u / (len(a) * len(b)) - 1
    pair.append({"변수": name, "L3 중앙값": a.median(), "L5 중앙값": b.median(),
                 "순위이연상관": rbc, "p": p})
pd.DataFrame(pair).set_index("변수")

**읽는 법** — 순위이연상관이 음수면 L3가 L5보다 작은 쪽, 양수면 큰 쪽입니다(−1~1).
L3는 **편의시설은 L5보다 적고 가격은 높습니다.** 두 방향이 반대라는 것이 L3의 모순입니다.

## 2. 로지스틱 회귀 — 무엇이 L3를 만드나

| 항목 | 설계 |
|---|---|
| 대상 | L3 + L4 + L5 · `source == 0` · 최소 숙박일 30박 미만 · `price_rel`·`amenity_rel`·`quote_peak` 보유 · **`L3_new` 제외** |
| 목표 | L3인가 (최근 1년 무거래) |
| 설명 변수 | 동급 대비 가격(로그) · 동급 대비 편의시설(5개 단위) · 최소 숙박일 구간(기준 2박) · 개인실 · 성수기 견적 |
| 뺀 변수 | 층 정의에 쓴 세 컬럼(순환 논리) · 슈퍼호스트(거래의 결과일 수 있음) · 평점(무거래 매물은 절반이 결측) |

최소 숙박일 30박 이상을 빼는 이유: L3는 정의상 30박 미만이라, 넣으면 "30박 이상 = L3 아님"을 모델이 공짜로 맞힙니다.

In [ ]:
# [2-1] 모델링 표본
m = df[(df["source"] == 0) & df["segment"].isin(["L3", "L4", "L5"]) & (df["minimum_nights"] < 30)
       & df["price_rel"].notna() & df["amenity_rel"].notna() & df["quote_peak"].notna()
       & ~df["L3_new"]].copy()
m["y"] = (m["segment"] == "L3").astype(int)
mn = pd.cut(m["minimum_nights"], [0, 1, 2, 3, 7, 29], labels=["1", "2", "3", "4-7", "8-29"])

X = pd.DataFrame({
    "동급 대비 가격 (log)":       np.log(m["price_rel"].clip(0.2, 5)),
    "동급 대비 편의시설 (+5개)":   m["amenity_rel"].clip(-40, 40) / 5,
    "최소 숙박 1박":             (mn == "1").astype(int),
    "최소 숙박 3박":             (mn == "3").astype(int),
    "최소 숙박 4-7박":           (mn == "4-7").astype(int),
    "최소 숙박 8-29박":          (mn == "8-29").astype(int),
    "개인실·호텔·공유":           (m["room_type"] != "Entire home/apt").astype(int),
    "성수기(6-8월) 견적":         m["quote_peak"].astype(int),
}, index=m.index)
y = m["y"]
print(f"표본 {len(m):,}건 · L3 비율 {y.mean():.1%} · 도시 {m['region'].nunique()}개")

In [ ]:
# [2-2] 적합 + 오즈비 95% 신뢰구간
# 규제 없는 로지스틱을 쓰고, 표준오차는 피셔 정보행렬 (X'WX)^-1 에서 직접 구한다.
def fit_logit(X, y):
    model = LogisticRegression(penalty=None, max_iter=5000).fit(X, y)
    p = model.predict_proba(X)[:, 1]
    Xd = np.column_stack([np.ones(len(X)), X.to_numpy(dtype=float)])
    cov = np.linalg.inv(Xd.T @ (Xd * (p * (1 - p))[:, None]))
    se = np.sqrt(np.diag(cov))[1:]
    return model, se

logit, se = fit_logit(X, y)
b = logit.coef_[0]
coef = pd.DataFrame({
    "계수": b, "표준오차": se, "z": b / se,
    "오즈비": np.exp(b), "95% 하한": np.exp(b - 1.96 * se), "95% 상한": np.exp(b + 1.96 * se),
}, index=X.columns)

# 표준화 계수 — 단위가 다른 변수끼리 크기를 비교하려고 각 변수를 표준편차 1로 맞춘 뒤 다시 적합
Z = (X - X.mean()) / X.std()
coef["표준화 계수"] = LogisticRegression(penalty=None, max_iter=5000).fit(Z, y).coef_[0]
coef.sort_values("표준화 계수", key=abs, ascending=False)

In [ ]:
# [2-3] 검증 — ① 무작위 7:3  ② 도시 단위 5겹 (학습에 쓰지 않은 도시에서 평가)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
auc_random = roc_auc_score(yte, LogisticRegression(penalty=None, max_iter=5000)
                           .fit(Xtr, ytr).predict_proba(Xte)[:, 1])

city_auc = []
for tr, te in GroupKFold(n_splits=5).split(X, y, groups=m["region"]):
    mdl = LogisticRegression(penalty=None, max_iter=5000).fit(X.iloc[tr], y.iloc[tr])
    city_auc.append(roc_auc_score(y.iloc[te], mdl.predict_proba(X.iloc[te])[:, 1]))

# 변수 하나만 넣었을 때의 AUC — 어느 쪽이 혼자서 더 잘 가르나
solo = {c: roc_auc_score(yte, LogisticRegression().fit(Xtr[[c]], ytr).predict_proba(Xte[[c]])[:, 1])
        for c in ["동급 대비 가격 (log)", "동급 대비 편의시설 (+5개)"]}

print(f"무작위 분할 AUC       {auc_random:.3f}")
print(f"도시 단위 5겹 AUC     {np.mean(city_auc):.3f}  (겹별 {', '.join(f'{a:.3f}' for a in city_auc)})")
for k, v in solo.items():
    print(f"단독 AUC · {k:<18} {v:.3f}")

In [ ]:
# [2-4] 오즈비 그림 — 1보다 크면 L3가 될 가능성↑, 작으면↓ (로그 축)
order = coef.sort_values("오즈비").index
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ypos = np.arange(len(order))
ax.errorbar(coef.loc[order, "오즈비"], ypos,
            xerr=[coef.loc[order, "오즈비"] - coef.loc[order, "95% 하한"],
                  coef.loc[order, "95% 상한"] - coef.loc[order, "오즈비"]],
            fmt="o", ms=8, color=C["main"], ecolor=C["main"], elinewidth=2, capsize=0)
ax.axvline(1, color=C["gray"], lw=1)
ax.set_xscale("log")
ticks = [0.5, 0.7, 1, 1.5, 2, 3, 4]
ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
ax.set_yticks(ypos, order)
ax.set_xlabel("오즈비 (L3가 될 가능성, 로그 축)")
ax.set_title("L3(단기 무거래)가 될 가능성을 바꾸는 요인", loc="left")
for yi, name in zip(ypos, order):
    ax.annotate(f"{coef.loc[name, '오즈비']:.2f}", (coef.loc[name, "95% 상한"], yi),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color="#444")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**읽는 법**
- **표준화 계수**는 "변수를 표준편차 하나만큼 움직였을 때의 효과"라 변수끼리 크기를 비교할 수 있습니다. 편의시설과 가격 중 어느 쪽 절댓값이 큰지가 발표 메시지입니다.
- **오즈비**는 해석용입니다. 예: 편의시설 오즈비 0.8 = 동급보다 5개 많을 때마다 L3가 될 오즈가 20% 줄어듦.
- 가격은 로그로 넣었으므로 오즈비는 "로그 1단위(= 2.72배)"당 값입니다. **동급보다 2배 비쌀 때의 오즈비는 `2 ** 계수`** 로 계산합니다.
- 성수기 견적의 오즈비가 1보다 작은 것은, 비수기 날짜로 견적이 잡힌 매물(성수기 예약이 이미 찬 매물이 아님)에 무거래가 더 많다는 뜻입니다. 이 변수는 계절 효과를 **통제하려고** 넣은 것이라 해석의 중심에 두지 않습니다.
- 이 모델은 **인과가 아니라 연관**을 보여줍니다. AUC가 0.7 안팎이면 예측 도구로는 보통이라 "요인 크기를 나누어 보는 도구"로 소개합니다.

## 3. 결정 트리 — 누구에게 개입하나

해석할 수 있도록 **변수 3개(동급 대비 가격 · 동급 대비 편의시설 · 최소 숙박일), 깊이 3, 잎 하나당 5,000건 이상**으로 제한합니다.
사람이 기준선을 정하지 않고 **트리가 데이터에서 찾은 분기점**을 알림 대상 선정 규칙으로 씁니다.

In [ ]:
# [3-1] 적합
T = pd.DataFrame({"price_rel": m["price_rel"], "amenity_rel": m["amenity_rel"],
                  "minimum_nights": m["minimum_nights"]})
Ttr, Tte = T.loc[Xtr.index], T.loc[Xte.index]
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5000, random_state=SEED).fit(Ttr, ytr)
print(f"트리 테스트 AUC {roc_auc_score(yte, tree.predict_proba(Tte)[:, 1]):.3f}")
print(export_text(tree, feature_names=list(T.columns), decimals=2))

In [ ]:
# [3-2] 잎별 L3 비율 — 규칙을 사람이 읽는 문장으로
def leaf_rules(tree, names):
    t = tree.tree_
    out = {}
    def walk(node, conds):
        if t.children_left[node] == -1:
            out[node] = " & ".join(conds) or "(전체)"
            return
        f, thr = names[t.feature[node]], t.threshold[node]
        walk(t.children_left[node], conds + [f"{f} ≤ {thr:.2f}"])
        walk(t.children_right[node], conds + [f"{f} > {thr:.2f}"])
    walk(0, [])
    return out

rules = leaf_rules(tree, list(T.columns))
leaf = pd.Series(tree.apply(T), index=T.index)
leaves = (pd.DataFrame({"leaf": leaf, "y": y})
          .groupby("leaf")["y"].agg(n="size", L3비율="mean")
          .assign(규칙=lambda d: d.index.map(rules),
                  전체대비=lambda d: d["L3비율"] / y.mean())
          .sort_values("L3비율", ascending=False))
print(f"전체 L3 비율 {y.mean():.1%}")
leaves

In [ ]:
# [3-3] 분기점 안정성 — 표본을 절반씩 5번 다시 뽑아도 같은 분기점이 나오나
def splits(tree, names):
    t = tree.tree_
    return [(names[t.feature[i]], round(t.threshold[i], 2))
            for i in range(t.node_count) if t.children_left[i] != -1]

stab = []
for s in range(5):
    idx = T.sample(frac=0.5, random_state=100 + s).index
    tr_ = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5000, random_state=SEED).fit(T.loc[idx], y.loc[idx])
    sp = splits(tr_, list(T.columns))
    stab.append({"표본": s + 1,
                 "가격 분기점": sorted({v for f, v in sp if f == "price_rel"}),
                 "편의시설 분기점": sorted({v for f, v in sp if f == "amenity_rel"}),
                 "최소숙박 분기점": sorted({v for f, v in sp if f == "minimum_nights"})})
pd.DataFrame(stab).set_index("표본")

In [ ]:
# [3-4] 트리 그림
fig, ax = plt.subplots(figsize=(15, 6.5))
plot_tree(tree, feature_names=["동급 대비 가격", "동급 대비 편의시설", "최소 숙박일"],
          class_names=["거래", "L3"], filled=False, impurity=False, proportion=True,
          rounded=True, precision=2, fontsize=8, ax=ax)
ax.set_title("L3 판별 결정 트리 (깊이 3) — value = [거래, L3] 비율", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# [3-5] 트리 분기점으로 본 L3 비율 — 가격 × 편의시설 (최소 숙박일 4박 이하 / 5박 이상 나눠서)
pb = pd.cut(m["price_rel"], [0, .7, .85, 1, 1.15, 1.3, 1.6, 2.4, np.inf],
            labels=["<0.7", "0.7-0.85", "0.85-1", "1-1.15", "1.15-1.3", "1.3-1.6", "1.6-2.4", "2.4+"])
ab = pd.cut(m["amenity_rel"], [-np.inf, -13, -8, -3, 3, 8, np.inf],
            labels=["-13 미만", "-13~-8", "-8~-3", "±3", "+3~+8", "+8 초과"])
long_stay = np.where(m["minimum_nights"] >= 5, "최소 숙박 5박 이상", "최소 숙박 4박 이하")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, key in zip(axes, ["최소 숙박 4박 이하", "최소 숙박 5박 이상"]):
    sel = long_stay == key
    piv = pd.crosstab(ab[sel], pb[sel], values=y[sel], aggfunc="mean")
    im = ax.imshow(piv.to_numpy(), cmap="Blues", vmin=0, vmax=0.8, aspect="auto")
    ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=30, ha="right")
    ax.set_yticks(range(piv.shape[0]), piv.index)
    ax.set_xlabel("동급 대비 가격 (배)")
    ax.set_title(key, loc="left")
    ax.grid(False)
axes[0].set_ylabel("동급 대비 편의시설 (개)")
fig.colorbar(im, ax=axes, label="L3 비율", shrink=0.9)
fig.suptitle("가격이 높고 편의시설이 적을수록, 최소 숙박일이 길수록 L3 비율이 높다", x=0.01, y=1.04, ha="left")
plt.show()

In [ ]:
# [3-6] 위 그림의 표 — 최소 숙박 4박 이하
pd.crosstab(ab[long_stay == "최소 숙박 4박 이하"], pb[long_stay == "최소 숙박 4박 이하"],
            values=y[long_stay == "최소 숙박 4박 이하"], aggfunc="mean").round(2)

## 결과 요약

실행 후 아래 수치를 [`reports/MODELING_PLAN.md`](../reports/MODELING_PLAN.md) 2·3단계 결과 칸에 옮깁니다.

| 확인할 것 | 셀 |
|---|---|
| 층 차이의 효과 크기(ε², Cramér V) | `[1-2]` |
| 편의시설 vs 가격 표준화 계수 | `[2-2]` |
| 무작위 / 도시 단위 AUC | `[2-3]` |
| 트리 분기점과 안정성 | `[3-1]` `[3-3]` |
| 최악 조합 잎의 L3 비율 | `[3-2]` |